# HeatingBits Jupyter notebook

This is a notebook to consolidate the work developed in the project HeatingBits.

Five main scenarios will be evaluated throughout the project:

- BAU (Business As Usual) with heat recovered in the cold source of the HP
- Heat recovery for heating and electricity production
- Heat conversion to electricity with an ORC with heat recovery from the condenser
- Heat storage with grid coordination for CO2 mitigation
- Grid coordination by data production management and heat storage

## Optimization

Improvements:

In order to perform the clustering for weekdays and emissions when using a custom weather data, it is necessary to include the following two columns in the CSV file: 'Weekday' and 'Emissions'. In the future, when we want to perform the clustering for the DC's heat load, we can check Sai's and Theresa's implementation.




In [11]:
from reho.model.reho import *
from reho.plotting import plotting
from pathlib import Path


In [16]:
data_centre_heat_profile = pd.read_csv(str(Path.cwd() / 'data' / 'clustering' / 'DL_Pully_10_24_T_I_E_D_test.dat'), header=None).to_numpy()
print(data_centre_heat_profile)

[[46.77639766]
 [44.91044259]
 [43.04448751]
 [41.17853243]
 [39.31257735]
 [37.44662227]
 [35.5806672 ]
 [34.88175752]
 [34.29066549]
 [33.69957347]
 [33.10848144]
 [31.37002138]
 [30.17719204]
 [28.9843627 ]
 [27.00728825]
 [26.50056867]
 [25.99384909]
 [31.14781182]
 [47.45120571]
 [47.70357939]
 [47.95595306]
 [46.65187828]
 [44.93905044]
 [39.80056692]
 [38.08773908]
 [36.37491125]
 [34.66208341]
 [32.97323419]
 [33.11712137]
 [28.57700101]
 [27.72398494]
 [26.87096887]
 [28.13777688]
 [31.06260213]
 [33.98742738]
 [36.91225263]
 [39.83707788]
 [42.76190313]
 [47.25184075]
 [45.66274745]
 [44.07365415]
 [42.48456084]
 [40.89546754]
 [39.30637424]
 [37.71728094]
 [36.12818763]
 [34.53909433]
 [37.82198208]
 [35.58231947]
 [36.19342704]
 [36.80453461]
 [37.41564218]
 [38.02674975]
 [38.63785732]
 [38.82267419]
 [39.96127349]
 [41.09987279]
 [42.23847209]
 [43.37707139]
 [44.5156707 ]
 [45.65427   ]
 [46.7928693 ]
 [47.37104221]
 [45.7221971 ]
 [44.073352  ]
 [42.42450689]
 [40.77566


Start first scenario:

In [10]:
 # Set path to CSV buildings data
buildings_filename = str(Path.cwd() / 'data' / 'EPFL_3.csv')

# Set building parameters
reader = QBuildingsReader()
n_house = 1  # 53 buildings in total
qbuildings_data = reader.read_csv(buildings_filename=buildings_filename, nb_buildings=n_house)

# Select clustering options for weather data, with custom profiles
custom_weather_profile = str(Path.cwd() / 'data' / 'profiles' / 'pully_v2.csv')

# Select clustering options for weather data
cluster = {'custom_weather': custom_weather_profile,
           'Location': 'Pully',
           'Attributes': ['T', 'I', 'W', 'E'],
           'Periods': 10,
           'PeriodDuration': 24}

# Set scenario
scenario = dict()
scenario['Objective'] = 'GWP'
scenario['name'] = 'GWP'
scenario['exclude_units'] = ['HeatPump_Air','HeatPump_Lake','HeatPump_Anergy','HeatPump_DHN', 'ElectricalHeater_SH', 'ThermalSolar'] #'OIL_Boiler',  'NG_Boiler','HeatPump_Air', 'HeatPump_Lake''HeatPump_Anergy''DataHeatSH', #['ThermalSolar', 'Battery', 'NG_Boiler']
scenario['enforce_units'] = ['HeatPump_Geothermal_district']
scenario["specific"] = ['enforce_PV_max'] #'enforce_DHN'

# Initialize available units and grids
grids = infrastructure.initialize_grids({'Electricity': {"Cost_demand_cst": 0.08, "Cost_supply_cst": 0.08},
                                         'Heat': {"Cost_demand_cst": 0.0001, "Cost_supply_cst": 0.0002,  "GWP_supply_cst": 0}})
units = infrastructure.initialize_units(scenario, grids, district_data=True)
parameters = {'n_vehicles': np.array([0.0]), 'T_DHN_supply_cst': np.repeat(70.0, n_house),'T_DHN_return_cst': np.repeat(60.0, n_house), "TransformerCapacity": np.array([1e8, 0])}

# Set method options
method = {'building-scale': True,
          'save_stream_t': True,
          'use_dynamic_emission_profiles': True,
          'save_streams': True,
          'ORC_all_the_time': False}
#DW_params = {'max_iter': 2}

# Run optimization
reho = REHO(qbuildings_data=qbuildings_data, units=units, grids=grids, cluster=cluster, parameters=parameters,
            scenario=scenario, method=method, solver="gurobi") #DW_params=DW_params if district-scale: True
reho.single_optimization()

INITIATION, Iter:0 Pareto_ID: 0
Loaded .env from package root: /Users/eduardo/REHO_local/REHO/reho/.env
INITIATE HOUSE: Building1
Gurobi 10.0.2:   tech:nodefilestart = 0.5
Gurobi 10.0.2: optimal solution; objective -143482.4446
2665 simplex iterations
1 branching nodes
absmipgap=1.74623e-10, relmipgap=1.21703e-15

"option abs_boundtol 4.547473508864641e-13;"
or "option rel_boundtol 4.3941513913339447e-16;"
will change deduced dual values.

MASTER INITIATION, Iter:0
Gurobi 10.0.2:   tech:nodefilestart = 0.5
Gurobi 10.0.2: optimal solution; objective -1475090529
0 simplex iterations
absmipgap=53110.9, relmipgap=3.60052e-05
            GWP
0 -1.475091e+09
Empty DataFrame
Columns: []
Index: []
LAST MASTER ITERATION, Iter:1 Pareto_ID: 0
Gurobi 10.0.2:   tech:nodefilestart = 0.5
Gurobi 10.0.2: optimal solution; objective -1475090529
0 simplex iterations
absmipgap=53110.9, relmipgap=3.60052e-05
            GWP
0 -1.475091e+09
               Costs_op      Costs_inv  ...  EMOO_grid      Objecti

In [12]:
# Save results
reho.save_results(format=['xlsx', 'pickle'], filename=scenario['name'])

# Plot results
#plotting.plot_performance(reho.results, plot='costs', indexed_on='Scn_ID', label='EN_long',
#                          title="Economical performance").show()

#plotting.plot_performance(reho.results, plot='gwp', indexed_on='Scn_ID', label='EN_long',
#                          title="GWP performance").show()

plotting.plot_sankey(reho.results['GWP'][0], label='EN_long', color='ColorPastel').show()

Results are saved in results/GWP.pickle
Results are saved in results/GWP_GWP.xlsx


In [ ]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

import plotly.io as pio
pio.templates.default = "plotly"

results = reho.results

# Performance : Economical, Environmental, and Combined
plotting.plot_performance(results, plot='costs', indexed_on='Scn_ID', title="Economical performance").show()
plotting.plot_performance(results, plot='gwp', indexed_on='Scn_ID', title="Environmental performance").show()
plotting.plot_performance(results, plot='combined', indexed_on='Scn_ID', title="Combined performance (Economical + Environmental)").show()

# Costs and Revenues
plotting.plot_expenses(results, plot='costs', indexed_on='Scn_ID', title="Costs and Revenues").show()

# Sankey diagram
plotting.plot_sankey(results['totex'][0], label='EN_long', color='ColorPastel', title="Sankey diagram").show()

# Eud-use demands
plotting.plot_eud(results, label='EN_long', title="End-use demands").show()

# Energy profiles
units_to_plot = ['ElectricalHeater', 'HeatPump', 'PV', 'NG_Boiler']
plotting.plot_profiles(results['totex'][0], units_to_plot, label='EN_long', color='ColorPastel', resolution='weekly', title="Energy profiles with a weekly moving average").show()